# Intro 

The main objectives of this report is to see how the tensor products behaves throught the simulations and how it preforms acoutdning diferent wheights matrix 

In [ ]:
import os 
os.chdir("..")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import SplineTransformer
from sklearn.linear_model import Ridge
import statsmodels.api as sm
from itertools import product
import pandas as pd
from patsy import dmatrix
from st_repl import SpatialReg
import patsy

sr = SpatialReg()


In [ ]:
gdf = sr.spatial_panel(time=100,rho=0.7, seed=787)
gdf

In [ ]:
# Tensor model 

formula= "y_true ~ X_1 + X_2 + X_3 + te(cr(lat, df=6), cr(lon, df=6), constraints='center')"

# 1. Generate design matrices
y, design = patsy.dmatrices(formula, gdf)

# 2. Fit the Ridge model
model = Ridge(alpha=0.01)
model.fit(design, y)

# 3. Predict the trend/spatial surface
gdf['y_tensor'] = model.predict(design)


In [ ]:
for m in ["w_rook", "w_queen", "w_knn6"]:
    xb = gdf[["X_1","X_2","X_3",m]].values.reshape(-1,4)
    y_d = gdf["y_true"].values.reshape(-1,1)
    X = sm.add_constant(xb)
    results = sm.OLS(y_d, X).fit()
    gdf[f"ols_{m.split("_")[1]}"] = results.predict(X)

In [ ]:
gdf